# HW2 — <your dataset>

| Student | Group | Dataset | Submitted |
|---|---|---|---|
| | | | |

This is a scaffold. Keep the part titles, fill the rest; delete nothing you did not write.

## Setup

Runs locally from the `hw2/` folder. Put your dataset into `MY_DATASET`; the cost ratio follows from it. In Colab, also uncomment the three lines below it.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator

MY_DATASET = "..."     # <- your dataset, from the direct message
DATA = Path("data")

# --- Google Colab? Uncomment these three lines: ---
# !git clone https://github.com/seemsGoodNow/hse-data-science-course.git course
# DATA = Path("course/homeworks/hw2/data")
# %pip install catboost

DATASET_FILES = {
    "credit_card_default": "credit_card_default/credit_card_default.csv",
    "bank_marketing": "bank_marketing/bank_marketing.csv",
    "home_equity": "home_equity/hmeq.csv",
    "lending_club": "lending_club/lending_club_sample.csv",
    "hotel_bookings": "hotel_bookings/hotel_bookings.csv",
}
# a missed "yes" costs this many false alarms (README, "Your dataset")
COST_RATIOS = {
    "credit_card_default": 4,
    "bank_marketing": 6,
    "home_equity": 4,
    "lending_club": 4,
    "hotel_bookings": 0.5,
}
COST_RATIO = COST_RATIOS[MY_DATASET]


def make_ax_better(ax, minor=""):
    """Tidy one chart: drop the top/right frame, add a light grid.

    minor="x", "y" or "xy" also adds small ticks between the labelled ones.
    """
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    if "x" in minor:
        ax.xaxis.set_minor_locator(AutoMinorLocator())
    if "y" in minor:
        ax.yaxis.set_minor_locator(AutoMinorLocator())
    if minor:
        ax.tick_params(which="minor", length=2.5)
        ax.tick_params(which="major", length=5)
        ax.grid(which="minor", linewidth=0.15, color="tab:grey", alpha=0.25)
    ax.grid(linewidth=0.5, color="tab:grey", alpha=0.25)


records = pd.read_csv(DATA / DATASET_FILES[MY_DATASET])
print(
    f"{MY_DATASET}: {len(records):,} rows, "
    f"{records.shape[1]} columns, cost ratio {COST_RATIO}:1"
)

## Before you start: your data is not Taiwan

The bankruptcy table from Session 5 was all numbers, with no gaps. Yours has text columns, gaps, or both. Session code copied as is will stop with an error, or run and print wrong numbers.

- **The target as 1 and 0.** Models and the session counters expect 1 and 0. A text target doesn't always crash: a counter that compares with 1 just prints zeros. Convert it first, for example `(records["your_target"] == "yes").astype(int)` with your target column's name.
- **Text columns.** Logistic regression needs numbers (Part 3 shows how). CatBoost reads text itself, but only in the columns you name: `.fit(X_train, y_train, cat_features=text_columns)`, and the same list in `Pool(...)` for SHAP. If an assistant writes this code for you, tell it which columns are text.
- **Gaps.** Logistic regression stops at any missing value, with `Input X contains NaN` or `exog contains inf or nans`. The second message also appears when you divide a constant column by its std of zero. CatBoost accepts gaps in number columns; fill the gaps in text columns with a word, for example `.fillna("missing")`.

## Part 0 — Clean the data
Goal: a table you can trust, and a short list of every change you made to it.

Check four things, using the documentation page:

1. **Leakage.** When does each group of columns become known: before the outcome or after it? A column known only after the outcome must go, however good it makes the model look (Session 6).
2. **Codes.** Values the documentation doesn't explain, and numbers that are really category codes.
3. **Gaps.** Which columns have them, and whether rows with a gap say "yes" more often than rows without.
4. **Duplicates**, and columns that almost never change.

Show: the list "what I drop or fix, and why", one line per decision.

In [ ]:
# TODO

**What I drop or fix, and why (one line each):** …

## Part 1 — Explore: what goes with "yes"?
Goal: candidate drivers of the target, found with charts before any model.

1. The share of "yes", and the accuracy of a "model" that always says "no".
2. For 4 to 6 number columns you expect to matter: the "yes" rows and the "no" rows as two histograms on one chart, as in Session 3.
3. For 2 to 4 text columns: the share of "yes" in each category, with the number of rows behind it.
4. One more chart of your own choice that taught you something.

Every chart gets a title that says what is drawn, axis labels with units, and one sentence underneath on what you see.

Show: 3 to 6 hypotheses of the form "more X goes with more 'yes', because …". Part 3 tests them.

In [ ]:
# TODO

**My hypotheses (more X goes with more "yes", because …):** …

## Part 2 — Class weights: on or off?
Goal: a conscious choice about the rare class, made before any model.

Session 5 showed one cure for a rare "yes": class weights, where each "yes" row counts for more. In scikit-learn it is `class_weight="balanced"`, in CatBoost `auto_class_weights="Balanced"`. Decide whether you use them and write why. On a dataset that is barely imbalanced, "off" is a fair choice.

Use the same choice for every model you compare in Part 4. Part 5 checks whether it mattered.

Show: the choice and your reason, before any model.

**Class weights: on / off, and why (before any model):** …

## Part 3 — A baseline you can read
Goal: a logistic regression you can explain line by line.

1. Take 3 to 6 features from your Part 1 hypotheses, each with the direction you expect.
2. Split into train and test once, and keep this split for all later parts.
3. A text feature becomes 0/1 columns: `pd.get_dummies(table, columns=[...], drop_first=True, dtype=float)`. `drop_first` removes one category, because together they always add up to 1: an exact copy of the constant, the "exact twins" of Session 5. Each remaining weight then reads "compared with the dropped category".
4. Standardize the number features with the training mean and std, fit `sm.Logit`, and read the coefficients and their p-values. Read significance without class weights, even if you chose them in Part 2: statsmodels has no weights, and that is fine for reading directions. With tens of thousands of rows almost every p-value is tiny, so read the size of each coefficient too (Session 5).

Show: which directions match your hypotheses, which don't, and why.

If a weight is near zero, a p-value is near 1, or statsmodels says it did not converge, look at the share of "yes" for each value of that feature. A value that always means "yes", or always "no", breaks logistic regression, and it is a finding in itself.

**My features and the direction I expect for each:** …

In [ ]:
# TODO

**Which directions match, which don't, and why:** …

## Part 4 — Can boosting beat it?
Goal: find out whether a model you can't read aloud is worth it on your data.

1. Train CatBoost on all the features you kept in Part 0, on the same split. Its opponent is your baseline refitted in scikit-learn (`LogisticRegression`, as in Session 5), and both get the same Part 2 choice. A logistic regression on more features is a fair opponent too.
2. On the test part, compare the models by caught, missed and false alarms, then accuracy, recall and precision. Put the train numbers next to them: a big gap between train and test means the model memorised.

Show: does boosting win on your data, by how much, and is the gain worth losing a model you can read? On some datasets it wins clearly, on others it doesn't. Both are valid findings.

In [ ]:
# TODO

**Does boosting win on my data? Is the gain worth it?** …

## Part 5 — The threshold that costs least
Goal: the cut-off the business should use, given your cost ratio.

1. Take the model you would actually use, and say which.
2. Explain why accuracy misleads on your data, or why it misleads less. Show the confusion matrix at 0.5.
3. For thresholds from 0.01 to 0.99, compute the expected cost: missed × cost ratio + false alarms (Session 6). Draw one chart of cost against threshold, with the minimum and 0.5 marked.
4. At the chosen threshold, report recall, precision and the share of rows flagged.
5. Did your Part 2 choice matter? Fit the same model with the opposite choice (weights on instead of off, or the other way round), find its best threshold, and compare the two thresholds and the two costs.

Show: the threshold, what it costs compared with 0.5, what the business does with a flagged row, and whether your Part 2 choice mattered. Choosing the threshold on the test part, as in Session 6, is accepted; cross-validation on the training part is stronger. Say which you did.

In [ ]:
# TODO

**Model:** …

**Threshold, its cost against 0.5, and what the business does with a flagged row:** …

**Did the Part 2 choice matter? Best threshold and cost, weights on vs off:** …

**Threshold chosen on:** test part / cross-validation on the training part

## Part 6 — What drives the target
Goal: two views of "what matters", and two cases explained.

1. Put your Part 3 coefficients next to the boosting model's SHAP values: an importance ranking (built-in or mean |SHAP|, say which) and the beeswarm, as in Session 5. Do they agree on the main drivers and their direction? Where they don't, why: twins, a non-linear effect, gaps? Compare with your Part 1 charts too.
2. The beeswarm colours only mean something for number columns. For a text column, average its SHAP values by category, with the number of rows in each.
3. Explain two rows, one the model flags and one it clears, in words a manager would accept. Before each sentence, check the feature values behind the bars: a bar says which feature pushed, the value says why.

In [ ]:
# TODO

**Coefficients vs SHAP: where they agree, where they don't, and why:** …

**The flagged row, in a manager's words:** …

**The cleared row, in a manager's words:** …

## Part 7 — Memo to the decision maker
At most 10 sentences, no code, for the person who owns the decision in your business (a credit committee, a head of sales, a revenue manager): what the model can and cannot do, which threshold you recommend and what it costs, the top-3 drivers, and what you'd want next (data, checks, monitoring).

≤ 10 sentences, no code.

**Memo:**